# 1/2 - Train the denoiser (Kaggle GPU)

Training only. The benchmark is a separate notebook because together they do
not fit in Kaggle's 12 h limit: run 1 spent 8.5 h training and was killed
partway through the grid.

## Before running
1. *Add Input* -> your `pointdenoise-code` dataset
2. *Add Input* -> your `pointdenoise-data` dataset
3. *Settings* -> *Accelerator* -> **GPU T4 x2**
4. *Settings* -> *Persistence* -> **Files only**

## After running
Download `best.pt` and `history.json`. Upload `best.pt` as a small Kaggle
dataset, then run `benchmark_pointdenoise.ipynb` against it.


In [ ]:
import glob, os, subprocess, sys

def find_dir(marker, root="/kaggle/input"):
    for base, dirs, files in os.walk(root):
        if marker in dirs or marker in files:
            return base
    return None

CODE = find_dir("pointdenoise")
DATA = find_dir("examples")
print("code:", CODE)
print("data:", DATA)
assert CODE, f"pointdenoise package not found. /kaggle/input holds: {os.listdir('/kaggle/input')}"
assert DATA, "benchmark data not found (looking for an 'examples' directory)"

sys.path.insert(0, CODE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "trimesh", "rtree"], check=False)

import torch
print("\ntorch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - turn it on in Settings")


## Noise is sampled per patch, not fixed

Run 1 trained at a fixed 2% and scored +56% CD at 2%, +73% at 3%, and only
+2% at 1%: it had learned one correction size and applied it regardless, so at
low noise it moved points that were already close. On a sphere the same model
came out 84% *worse* than the noisy input at 1%. Sampling the range the
benchmark actually tests fixes that and is better at every level.


In [ ]:
from pointdenoise.benchmark import load_training_clouds
from pointdenoise.data import Shape
import numpy as np

train_clouds = load_training_clouds(DATA, "PUNet", "sparse")
shapes = [Shape(pts, noise_level=0.02, rng=np.random.default_rng(i))
          for i, (_, pts) in enumerate(train_clouds)]
print(f"{len(shapes)} training shapes, {shapes[0].clean.shape[0]} points each")


In [ ]:
from pointdenoise.engine import train

model, history = train(
    shapes,
    out_dir="/kaggle/working/runs",
    epochs=60,
    batch_size=32,
    points_per_patch=256,
    patches_per_shape=1000,
    lr=1e-3,
    repulsion_weight=0.05,
    noise_range=(0.005, 0.03),   # the fix; None reproduces run 1
    model_kwargs={"d_model": 256, "num_heads": 8, "num_layers": 6},
    num_workers=2,
    seed=0,
)


In [ ]:
import matplotlib.pyplot as plt, shutil

fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4))
a.plot([h["total"] for h in history], label="total")
a.plot([h["chamfer"] for h in history], label="chamfer")
a.set_xlabel("epoch"); a.set_ylabel("loss"); a.legend(); a.grid(alpha=.3)
a.set_title("Training loss")
b.plot([h["lr"] for h in history], color="tab:orange")
b.set_yscale("log"); b.set_xlabel("epoch"); b.set_ylabel("lr"); b.grid(alpha=.3)
b.set_title("Learning rate")
plt.tight_layout(); plt.savefig("/kaggle/working/loss.png", dpi=120); plt.show()

print(f"epoch 1 {history[0]['total']:.6f} -> epoch {len(history)} {history[-1]['total']:.6f}")
print(f"total {sum(h['seconds'] for h in history)/3600:.1f} h")
shutil.copy("/kaggle/working/runs/best.pt", "/kaggle/working/best.pt")
print("\ndownload best.pt and history.json, then run the benchmark notebook")
